In [155]:
import pandas as pd
import plotly.express as px
import numpy as np
from matplotlib.pyplot import legend
from scipy.stats import chi2_contingency
import math
import db
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go

In [156]:
data = db.Database()
sources = pd.DataFrame(data.get_source_papers())
papers = data.get_all_ref_papers()

In [157]:
random_papers = sources[sources["Journal"] == "European journal of psychotraumatology"].sample(n=60, random_state=302010)
random_papers_ids = random_papers["_id"].to_list()
sources = sources[~sources["_id"].isin(random_papers_ids)]

In [158]:
sources["Open Access"].value_counts()

Open Access
True     110
False     72
Name: count, dtype: int64

In [159]:
sources["OA Standard"].value_counts()

OA Standard
closed    72
gold      65
hybrid    31
green      9
bronze     5
Name: count, dtype: int64

In [160]:
print(sources[sources["Open Access"]]["Journal"].value_counts())
print(sources[sources["Open Access"]]["Journal"].value_counts().sum())
print(sources[sources["Open Access"] == False]["Journal"].value_counts())
print(sources[sources["Open Access"] == False]["Journal"].value_counts().sum())
#sources["Journal"].value_counts()

Journal
European journal of psychotraumatology        38
Journal of Experimental Psychology General    17
Australian Journal of Psychology              16
Cogent Psychology                             10
Journal of Sports Sciences                     9
Neuropsychology                                8
Traumatology An International Journal          8
Evolutionary Behavioral Sciences               2
Sport Exercise and Performance Psychology      2
Name: count, dtype: int64
110
Journal
Traumatology An International Journal         27
Evolutionary Behavioral Sciences              16
Journal of Experimental Psychology General    16
Neuropsychology                                9
Sport Exercise and Performance Psychology      4
Name: count, dtype: int64
72


In [161]:
open_ground = []
closed_ground = []
ref_papers = data.get_all_ref_papers_grouped_by_source()


for source in sources.to_dict(orient="records"):
    ref_paper = ref_papers.get(source["_id"], [])
    if source["Open Access"]:
        for paper in ref_paper:
            open_ground.append(paper)
    else:
        for paper in ref_paper:
            closed_ground.append(paper)
open_ground = pd.DataFrame(open_ground)
closed_ground = pd.DataFrame(closed_ground)


In [162]:
exclude_paper_ids = []
for source_id, group in ref_papers.items():
    #print(f"Source ID: {source_id}")
    fin_group = pd.DataFrame(group)
    if source_id in random_papers_ids:
        for paper in fin_group.to_dict(orient="records"):
            exclude_paper_ids.append(paper["_id_x"])
papers = papers[~papers["_id"].isin(exclude_paper_ids)]

In [163]:
ids_second_iter = sources["_id"].sample(n=30, random_state=302010)
papers_second_iter = []
for source_id in ids_second_iter:
    ref_paper = ref_papers.get(source_id, [])
    for paper in ref_paper:
        papers_second_iter.append(paper)
papers_second_iter = pd.DataFrame(papers_second_iter)

In [164]:
closed_ground["Open Access"].value_counts()

Open Access
False    1045
True      769
Name: count, dtype: int64

In [165]:
open_ground["Open Access"].value_counts()

Open Access
True     1453
False    1223
Name: count, dtype: int64

In [166]:
closed_ground["Citation Count"].describe()

count     1820.000000
mean       312.358242
std        883.173060
min          0.000000
25%         21.000000
50%         60.000000
75%        198.000000
max      11295.000000
Name: Citation Count, dtype: float64

In [167]:
open_ground["Citation Count"].describe()

count     2699.000000
mean       440.287143
std       2479.425862
min          0.000000
25%         20.000000
50%         59.000000
75%        205.000000
max      80355.000000
Name: Citation Count, dtype: float64

In [168]:
generated_papers_iter0 = data.get_all_gen_papers_grouped_by_source_and_llm()
generated_papers_iter1 = data.get_all_gen_papers_grouped_by_source_and_llm(1)
generated_papers_iter2 = data.get_all_gen_papers_grouped_by_source_and_llm(2)

llms = ["ChatGPT", "Claude", "Gemini"]

In [169]:
chatgpt_papers_iter0 = []
claude_papers_iter0 = []
gemini_papers_iter0 = []

for (source_id, llm), gen in generated_papers_iter0.items():
    if source_id in random_papers_ids:
        continue
    gen_paper_iter0 = generated_papers_iter0.get((source_id, llm), [])
    gen_paper_iter0 = pd.DataFrame(gen_paper_iter0)
    gen_paper_iter0["SourceID"] = source_id
    gen_paper_iter0 = gen_paper_iter0.to_dict(orient="records")
    if llm == "ChatGPT":
        chatgpt_papers_iter0.extend(gen_paper_iter0)
    elif llm == "Claude":
        claude_papers_iter0.extend(gen_paper_iter0)
    elif llm == "Gemini":
        gemini_papers_iter0.extend(gen_paper_iter0)

chatgpt_papers_iter0 = pd.DataFrame(chatgpt_papers_iter0)
claude_papers_iter0 = pd.DataFrame(claude_papers_iter0)
gemini_papers_iter0 = pd.DataFrame(gemini_papers_iter0)

In [170]:
stacked_data = []
hallucination_threshold = 2
for llm, df in [("ChatGPT 5.2", chatgpt_papers_iter0), ("Claude Opus 4.6", claude_papers_iter0), ("Gemini 3.1 Flash Lite", gemini_papers_iter0)]:
    oa_counts = df[(df["Hallucination"] <= hallucination_threshold) & (df["Hallucination"] != -1)]["Open Access"].value_counts()
    hallucination_count = df[df["Hallucination"] > hallucination_threshold].shape[0]
    no_doi = df[df["Hallucination"] == -1].shape[0]
    stacked_data.append({
        "LLM": llm,
        "Open Access": oa_counts.get(True, 0),
        "Closed Access": oa_counts.get(False, 0),
        "Hallucination": hallucination_count,
        "No DOI": no_doi
    })
ground_oa_counts = papers["Open Access"].value_counts()

stacked_data.insert(0, {
    "LLM": "Ground Truth",
    "Open Access": ground_oa_counts.get(True, 0),
    "Closed Access": ground_oa_counts.get(False, 0),
    "No DOI": papers.shape[0] - ground_oa_counts.sum(),
    "Hallucination": 0
})

stacked_df = pd.DataFrame(stacked_data)

fig = px.bar(stacked_df, x="LLM", y=["Open Access", "Closed Access", "Hallucination", "No DOI"], title="Open Access Status by LLM", subtitle=f"Hallucination Threshold = {hallucination_threshold}", labels={"value": "Count", "LLM": "Source"}, text_auto=True, color_discrete_map={"Open Access": "lightgreen", "Closed Access": "salmon", "Hallucination": "lightblue", "No DOI": "lightgray"})
fig.update_legends(title_text='Status')
fig.show()

print(stacked_df)

                     LLM  Open Access  Closed Access  No DOI  Hallucination
0           Ground Truth         2222           2268     542              0
1            ChatGPT 5.2         1187           2170     738            937
2        Claude Opus 4.6         1369           2245     584            834
3  Gemini 3.1 Flash Lite          543           1253     833           2373


In [171]:

stacked_df_oa_ca = stacked_df[["LLM", "Open Access", "Closed Access"]]
stacked_df_oa_ca = stacked_df_oa_ca.melt(id_vars=["LLM"], value_vars=["Open Access", "Closed Access"], var_name="Status", value_name="Count")

stacked_df_oa_ca["Count"] = stacked_df_oa_ca.groupby("LLM")["Count"].transform(lambda x: x / x.sum() * 100).round(2)
fig = px.bar(stacked_df_oa_ca, x="LLM", y="Count", color="Status", title="Open Access vs Closed Access by LLM", subtitle=f"Hallucination Threshold = {hallucination_threshold}", labels={"Count": "Percentage", "LLM": "Source"}, text_auto=True, color_discrete_map={"Open Access": "lightgreen", "Closed Access": "salmon"})
fig.show()

In [172]:
oa_ref_papers = []
for source_group, SID in zip(ref_papers, ref_papers.keys()):
    for g_item in source_group:
        if isinstance(g_item["Open Access"], bool):
            g_item["SourceID"] = SID
            g_item["Model"] = "Ground Truth"
            oa_ref_papers.append(g_item)
oa_ref_papers = pd.DataFrame(oa_ref_papers)
combine_papers_iter0 = [[],[],[]]
exact_matches = [0,0,0]
for llm, df, x in [("ChatGPT 5.2", chatgpt_papers_iter0, 0), ("Claude Opus 4.6", claude_papers_iter0, 1), ("Gemini 3.1 Flash Lite", gemini_papers_iter0, 2)]:
    for oa_paper in oa_ref_papers.to_dict("records"):
        gen_paper = df.loc[(df["Index"] == oa_paper["Index"]) & (df["SourceID"] == oa_paper["SourceID"]) & (df["Hallucination"] < hallucination_threshold)]
        if len(gen_paper) > 0:
            gen_paper = gen_paper.to_dict("records")[0]
            if isinstance(gen_paper["Open Access"], bool):
                gen_paper["Model"] = llm
                gen_paper["OA"] = gen_paper["Open Access"]
                gen_paper["OAS"] = gen_paper["OA Standard"]
                oa_paper["OA"] = oa_paper["Open Access"]
                oa_paper["OAS"] = oa_paper["OA Standard"]
                combine_papers_iter0[x].append(gen_paper)
                combine_papers_iter0[x].append(oa_paper)
                if oa_paper["DOI"] == gen_paper["DOI"]:
                    exact_matches[x] += 1

print(exact_matches)

[203, 520, 121]


In [173]:
for llm, x in [("ChatGPT 5.2", 0), ("Claude Opus 4.6", 1), ("Gemini 3.1 Flash Lite", 2)]:
    df = pd.DataFrame(combine_papers_iter0[x])
    df_sub = df[["SourceID", "Model", "OA", "OAS", "Index"]]
    df_sub["SourceID"] = df_sub["SourceID"].astype(str)
    df_sub["Model"] = pd.Categorical(
        df_sub["Model"],
        categories=["Ground Truth", llm]
    )
    df_sub["OA"] = df_sub["OA"].astype(int)
    df_sub.to_csv("../Sub_Sets/Ground-" + llm.split(" ")[0] + "_Iter0.csv", index=False)


In [174]:
oa_gen_papers_iter0 = []
oa_gen_papers_iter0.append(chatgpt_papers_iter0[((chatgpt_papers_iter0["Open Access"] == True) | (chatgpt_papers_iter0["Open Access"] == False)) & (chatgpt_papers_iter0["Hallucination"] < hallucination_threshold)])
oa_gen_papers_iter0.append(claude_papers_iter0[((claude_papers_iter0["Open Access"] == True) | (claude_papers_iter0["Open Access"] == False)) & (claude_papers_iter0["Hallucination"] < hallucination_threshold)])
oa_gen_papers_iter0.append(gemini_papers_iter0[((gemini_papers_iter0["Open Access"] == True) | (gemini_papers_iter0["Open Access"] == False)) & (gemini_papers_iter0["Hallucination"] < hallucination_threshold)])

In [175]:
llms = ["ChatGPT 5.2", "Claude Opus 4.6", "Gemini 3.1 Flash Lite"]
combine_papers_llm_iter0 = [[],[],[]]
for i in range(3):
    base_llm = oa_gen_papers_iter0[i]
    for j in range(i+1, 3):
        comp_llm = oa_gen_papers_iter0[j]
        for base_paper in base_llm.to_dict("records"):
            comp_paper = comp_llm.loc[(comp_llm["Index"] == base_paper["Index"]) & (comp_llm["SourceID"] == base_paper["SourceID"])]
            if len(comp_paper) > 0:
                comp_paper = comp_paper.to_dict("records")[0]
                base_paper["Model"] = llms[i]
                comp_paper["Model"] = llms[j]
                base_paper["OA"] = base_paper["Open Access"]
                base_paper["OAS"] = base_paper["OA Standard"]
                comp_paper["OA"] = comp_paper["Open Access"]
                comp_paper["OAS"] = comp_paper["OA Standard"]
                combine_papers_llm_iter0[i+j-1].append(base_paper)
                combine_papers_llm_iter0[i+j-1].append(comp_paper)
        df = pd.DataFrame(combine_papers_llm_iter0[i+j-1])
        df_sub = df[["SourceID", "Model", "OA", "OAS", "Index"]]
        df_sub["SourceID"] = df_sub["SourceID"].astype(str)
        df_sub["Model"] = pd.Categorical(
            df_sub["Model"],
            categories=[llms[i], llms[j]]
        )
        df_sub["OA"] = df_sub["OA"].astype(int)
        df_sub.to_csv("../Sub_Sets/" + llms[i].split(" ")[0] + "-" + llms[j].split(" ")[0] + "_Iter0.csv", index=False)


In [176]:
open_access_matrix = [[0,0,0],[0,0,0],[0,0,0]]
standards_matrix =[[0,0,0,0], [0,0,0,0], [0,0,0,0], [0,0,0,0]]
excluded = 0
exact_matches = 0
currentLLM = "Gemini"
for currentLLM in ["Gemini", "ChatGPT", "Claude"]:
    for source in sources["_id"].unique():
        #print(source)
        ground_truth = pd.DataFrame(ref_papers.get(source, []))
        llm_results = pd.DataFrame(generated_papers_iter0.get((source, currentLLM), []))
        if len(ground_truth) == 0 or len(llm_results) == 0:
            continue
        ground_truth = ground_truth.sort_values("Index")
        #print(ground_truth)
        llm_results = llm_results.sort_values("Index")
        #print(llm_results)
        for gt_paper, llm_paper in zip(ground_truth.to_dict(orient="records"), llm_results.to_dict(orient="records")):
            gt_score = int(not gt_paper["Open Access"] if gt_paper["DOI"] != "NONE" else 2)
            llm_score = int(not llm_paper["Open Access"] if llm_paper["DOI"] != "NONE" and llm_paper["DOI"] is not None and float(str(llm_paper["Hallucination"])) < 3.5 else 2)
            open_access_matrix[gt_score][llm_score] += 1
            if gt_paper["DOI"] == llm_paper["DOI"]:
                exact_matches += 1

    if currentLLM == "Gemini":
        open_access_matrix[0][2] += 10
        open_access_matrix[1][2] += 11
        open_access_matrix[2][2] += 3

    if currentLLM == "Claude":
        open_access_matrix[0][2] += 9
        open_access_matrix[1][2] += 12

    print(f"Results for {currentLLM}:")
    print("Open Access Matrix: [Both OA, GT OA and LLM not OA, GT not OA and LLM OA, Both not OA]")
    print(open_access_matrix)
    print("Standards Matrix: Rows: GT OA Standard (Gold, Green, Bronze, Hybrid), Columns: LLM OA Standard (Gold, Green, Bronze, Hybrid)")
    print(standards_matrix)
    print(f"Excluded papers due to missing DOI: {excluded}")
    print(f"Exact matches: {exact_matches}")
    print("----------------------------------------------------------------")

Results for Gemini:
Open Access Matrix: [Both OA, GT OA and LLM not OA, GT not OA and LLM OA, Both not OA]
[[498, 1057, 667], [397, 1702, 702], [0, 0, 3]]
Standards Matrix: Rows: GT OA Standard (Gold, Green, Bronze, Hybrid), Columns: LLM OA Standard (Gold, Green, Bronze, Hybrid)
[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
Excluded papers due to missing DOI: 0
Exact matches: 123
----------------------------------------------------------------
Results for ChatGPT:
Open Access Matrix: [Both OA, GT OA and LLM not OA, GT not OA and LLM OA, Both not OA]
[[1295, 2268, 883], [968, 3684, 957], [0, 0, 3]]
Standards Matrix: Rows: GT OA Standard (Gold, Green, Bronze, Hybrid), Columns: LLM OA Standard (Gold, Green, Bronze, Hybrid)
[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
Excluded papers due to missing DOI: 0
Exact matches: 331
----------------------------------------------------------------
Results for Claude:
Open Access Matrix: [Both OA, GT OA and LLM not OA, GT not O

In [177]:
open_access_matrix_gemini = [0,0,0,0]
standards_matrix_gemini =[[0,0,0,0], [0,0,0,0], [0,0,0,0], [0,0,0,0]]
standards_ref_matrix = ["gold", "green", "bronze", "hybrid", "nan"]
for source in sources["_id"].unique():
    ground_truth = pd.DataFrame(ref_papers.get(source, []))
    llm_results = pd.DataFrame(generated_papers_iter0.get((source, "Gemini"), []))

    if llm_results.shape[0] == 0 or ground_truth.shape[0] == 0:
        continue
    ground_truth = ground_truth.sort_values("Index")
    llm_results = llm_results.sort_values("Index")

    for gt_paper, llm_paper in zip(ground_truth.to_dict(orient="records"), llm_results.to_dict(orient="records")):
        if not gt_paper["DOI"] or gt_paper["DOI"] == "NONE" or llm_paper["DOI"] is None or str(gt_paper["Open Access"]) == "nan" or str(llm_paper["Open Access"]) == "nan":
            continue
        if llm_paper["Open Access"] and gt_paper["Open Access"]:
            open_access_matrix_gemini[0] += 1
            if llm_paper["OA Standard"] == "nan" or gt_paper["OA Standard"] == "nan":
                continue
            llm_standard_score = int(standards_ref_matrix.index(str(llm_paper["OA Standard"])))
            gt_standard_score = int(standards_ref_matrix.index(str(gt_paper["OA Standard"])))

            standards_matrix_gemini[gt_standard_score][llm_standard_score] += 1
        elif gt_paper["Open Access"]:
            open_access_matrix_gemini[1] += 1
        elif llm_paper["Open Access"]:
            open_access_matrix_gemini[2] += 1
        else:
            open_access_matrix_gemini[3] += 1

In [178]:
stacked_data_oa_standard = []
for llm, df in [("ChatGPT 5.2", chatgpt_papers_iter0), ("Claude Opus 4.6", claude_papers_iter0), ("Gemini 3.1 Flash Lite", gemini_papers_iter0)]:
    oa_df = df[0 <= (df["Hallucination"] > hallucination_threshold) & (df["Open Access"] == True)]
    oa_standard_counts = oa_df["OA Standard"].value_counts()
    hallucination_count = df[df["Hallucination"] > hallucination_threshold].shape[0]
    stacked_data_oa_standard.append({
        "LLM": llm,
        "Gold": oa_standard_counts.get("gold", 0),
        "Green": oa_standard_counts.get("green", 0),
        "Bronze": oa_standard_counts.get("bronze", 0),
        "Hybrid": oa_standard_counts.get("hybrid", 0)
    })

ground_oa_standard_counts = papers[(papers["Open Access"] == True)]["OA Standard"].value_counts()
stacked_data_oa_standard.insert(0, {
    "LLM": "Ground Truth",
    "Gold": ground_oa_standard_counts.get("gold", 0),
    "Green": ground_oa_standard_counts.get("green", 0),
    "Bronze": ground_oa_standard_counts.get("bronze", 0),
    "Hybrid": ground_oa_standard_counts.get("hybrid", 0)
})
stacked_df_oa_standard = pd.DataFrame(stacked_data_oa_standard)

fig = px.bar(stacked_df_oa_standard, x="LLM", y=["Gold", "Green", "Bronze", "Hybrid"], title="OA Standard by LLM (Only for OA Papers)", subtitle=f"Hallucination Threshold = {hallucination_threshold}", labels={"value": "Count", "LLM": "Source", "variable": "Standard"}, text_auto=True, color_discrete_map={"Gold": "gold", "Green": "green", "Bronze": "peru", "Hybrid": "lightblue"})

fig.update_layout(
    legend=dict(
        title=dict(
            text="Standard"
        )
    )
)

fig.show()
print(stacked_df_oa_standard)

                     LLM  Gold  Green  Bronze  Hybrid
0           Ground Truth   744    606     432     440
1            ChatGPT 5.2   240    598     443     115
2        Claude Opus 4.6   368    581     435     220
3  Gemini 3.1 Flash Lite   166    383     305      94


In [179]:
oa_cols = ["Gold", "Green", "Bronze", "Hybrid"]

stacked_df_oa_standard_pct = stacked_df_oa_standard.copy()
stacked_df_oa_standard_pct[oa_cols] = (
    stacked_df_oa_standard_pct[oa_cols]
    .div(stacked_df_oa_standard_pct[oa_cols].sum(axis=1), axis=0)
    * 100
)

fig = px.bar(
    stacked_df_oa_standard_pct,
    x="LLM",
    y=oa_cols,
    title="OA Standard Distribution by LLM (Only OA Papers)",
    subtitle=f"Hallucination Threshold = {hallucination_threshold}",
    labels={"value": "Percentage (%)", "LLM": "Source", "variable": "Standard"},
    text_auto=".2f",
    color_discrete_map={
        "Gold": "gold",
        "Green": "green",
        "Bronze": "peru",
        "Hybrid": "lightblue"
    }
)

fig.update_layout(
    yaxis_title="Percentage (%)",
    yaxis_range=[0, 110],
    legend_title="Standard"
)

fig.show()

In [180]:
citation_counts = []
for llm, df in [("ChatGPT 5.2", chatgpt_papers_iter0), ("Claude Opus 4.6", claude_papers_iter0), ("Gemini 3.1 Flash Lite", gemini_papers_iter0)]:
    citation_counts.append(pd.DataFrame({"LLM": llm, "Citation Count": df["Citation Count"]}))

citation_counts.insert(0, pd.DataFrame({"LLM": "Ground Truth", "Citation Count": papers["Citation Count"]}))

citation_counts = pd.concat(citation_counts)
fig = px.box(
    citation_counts,
    x="LLM",
    y="Citation Count",
    points="all",
    title="Citation Count Distribution by LLM vs Ground Truth",
    labels={"Citation Count": "Citation Count", "LLM": "Source"},
    height=500,
    color="LLM"
)
fig.show()

In [181]:
citation_counts = []
for llm, df in [("ChatGPT 5.2", chatgpt_papers_iter0), ("Claude Opus 4.6", claude_papers_iter0), ("Gemini 3.1 Flash Lite", gemini_papers_iter0)]:
    citation_counts.append(pd.DataFrame({"LLM": llm, "Citation Count":  df["Citation Count"] + 1}))

citation_counts.insert(0, pd.DataFrame({"LLM": "Ground Truth", "Citation Count": papers["Citation Count"] + 1}))

citation_counts = pd.concat(citation_counts)
fig = px.box(
    citation_counts,
    x="LLM",
    y="Citation Count",
    points="all",
    title="Citation Count Distribution on a Log Scale by LLM vs Ground Truth",
    labels={"Citation Count": "Citation Count", "LLM": "Source"},
    color="LLM"
)
fig.update_yaxes(type='log', tickvals=[1, 10, 100, 1_000, 10_000, 100_000, 1_000_000], ticktext=['0', '10', '100', '1,000', '10,000', '100,000', '1,000,000'], range=[-0.2, 6])
fig.show()

In [182]:
chatgpt_papers_iter1 = []
claude_papers_iter1 = []
gemini_papers_iter1 = []

for (source_id, llm), gen in generated_papers_iter1.items():
    gen_paper_iter1 = generated_papers_iter1.get((source_id, llm), [])
    gen_paper_iter1 = pd.DataFrame(gen_paper_iter1)
    gen_paper_iter1["SourceID"] = source_id
    gen_paper_iter1 = gen_paper_iter1.to_dict(orient="records")
    if llm == "ChatGPT":
        chatgpt_papers_iter1.extend(gen_paper_iter1)
    elif llm == "Claude":
        claude_papers_iter1.extend(gen_paper_iter1)
    elif llm == "Gemini":
        gemini_papers_iter1.extend(gen_paper_iter1)

chatgpt_papers_iter1 = pd.DataFrame(chatgpt_papers_iter1)
claude_papers_iter1 = pd.DataFrame(claude_papers_iter1)
gemini_papers_iter1 = pd.DataFrame(gemini_papers_iter1)

In [183]:
stacked_data = []
for llm, df in [("Gemini 3.1 Flash Lite", gemini_papers_iter1), ("ChatGPT 5.2", chatgpt_papers_iter1), ("Claude Opus 4.6", claude_papers_iter1)]:
    oa_counts = df[(df["Hallucination"] <= hallucination_threshold) & (df["Hallucination"] != -1)]["Open Access"].value_counts()
    hallucination_count = df[df["Hallucination"] > hallucination_threshold].shape[0]
    no_doi = df[df["Hallucination"] == -1].shape[0]
    stacked_data.append({
        "LLM": llm,
        "Open Access": oa_counts.get(True, 0),
        "Closed Access": oa_counts.get(False, 0),
        "Hallucination": hallucination_count,
        "No DOI": no_doi
    })
ground_oa_counts = papers_second_iter["Open Access"].value_counts()

stacked_data.insert(0, {
    "LLM": "Ground Truth",
    "Open Access": ground_oa_counts.get(True, 0),
    "Closed Access": ground_oa_counts.get(False, 0),
    "No DOI": papers_second_iter.shape[0] - ground_oa_counts.sum(),
    "Hallucination": 0
})

stacked_df = pd.DataFrame(stacked_data)
fig = px.bar(stacked_df, x="LLM", y=["Open Access", "Closed Access", "Hallucination", "No DOI"], title="Open Access Status by LLM", subtitle=f"Hallucination Threshold = {hallucination_threshold}, second run of subset of 30 papers", labels={"value": "Count", "LLM": "Source", "variable": "Status"}, text_auto=True, color_discrete_map={"Open Access": "lightgreen", "Closed Access": "salmon", "Hallucination": "lightblue", "No DOI": "lightgray"})
fig.update_layout(
    legend=dict(
        title=dict(
            text="Status"
        )
    )
)
fig.show()

print(stacked_df)

                     LLM  Open Access  Closed Access  No DOI  Hallucination
0           Ground Truth          334            379      88              0
1  Gemini 3.1 Flash Lite           97            191     143            340
2            ChatGPT 5.2          179            342     139            141
3        Claude Opus 4.6          218            334      98            151


In [184]:
chatgpt_papers_iter0_comp = chatgpt_papers_iter0[chatgpt_papers_iter0["SourceID"].isin(ids_second_iter)]
gemini_papers_iter0_comp = gemini_papers_iter0[gemini_papers_iter0["SourceID"].isin(ids_second_iter)]
claude_papers_iter0_comp = claude_papers_iter0[claude_papers_iter0["SourceID"].isin(ids_second_iter)]

In [185]:
matches = 0
exact_matches = 0
for paper1 in chatgpt_papers_iter0_comp.to_dict(orient="records"):
    for paper2 in chatgpt_papers_iter1.to_dict(orient="records"):
        if paper1["SourceID"] == paper2["SourceID"]:
            if paper1["DOI"] == paper2["DOI"] and paper1["DOI"] != "NO DOI":
                matches += 1
                if paper1["Index"] == paper2["Index"]:
                    exact_matches += 1
print(f"Number of matches between iteration 0 and iteration 1: {matches}")
print(f"Number of exact matches between iteration 0 and iteration 1: {exact_matches}")
print(f"Number of papers in iteration 0: {len(chatgpt_papers_iter0_comp)}")
print(f"Number of papers in iteration 1: {len(chatgpt_papers_iter1)}")
print("----------------------------------------------")

matches = 0
exact_matches = 0
for paper1 in claude_papers_iter0_comp.to_dict(orient="records"):
    for paper2 in claude_papers_iter1.to_dict(orient="records"):
        if paper1["SourceID"] == paper2["SourceID"]:
            if paper1["DOI"] == paper2["DOI"] and paper1["DOI"] != "NO DOI":
                matches += 1
                if paper1["Index"] == paper2["Index"]:
                    exact_matches += 1
print(f"Number of matches between iteration 0 and iteration 1: {matches}")
print(f"Number of exact matches between iteration 0 and iteration 1: {exact_matches}")
print(f"Number of papers in iteration 0: {len(claude_papers_iter0_comp)}")
print(f"Number of papers in iteration 1: {len(claude_papers_iter1)}")
print("----------------------------------------------")

matches = 0
exact_matches = 0
for paper1 in gemini_papers_iter0_comp.to_dict(orient="records"):
    for paper2 in gemini_papers_iter1.to_dict(orient="records"):
        if paper1["SourceID"] == paper2["SourceID"]:
            if paper1["DOI"] == paper2["DOI"] and paper1["DOI"] != "NO DOI":
                matches += 1
                if paper1["Index"] == paper2["Index"]:
                    exact_matches += 1
print(f"Number of matches between iteration 0 and iteration 1: {matches}")
print(f"Number of exact matches between iteration 0 and iteration 1: {exact_matches}")
print(f"Number of papers in iteration 0: {len(gemini_papers_iter0_comp)}")
print(f"Number of papers in iteration 1: {len(gemini_papers_iter1)}")
print("----------------------------------------------")



Number of matches between iteration 0 and iteration 1: 324
Number of exact matches between iteration 0 and iteration 1: 242
Number of papers in iteration 0: 801
Number of papers in iteration 1: 801
----------------------------------------------
Number of matches between iteration 0 and iteration 1: 359
Number of exact matches between iteration 0 and iteration 1: 314
Number of papers in iteration 0: 801
Number of papers in iteration 1: 801
----------------------------------------------
Number of matches between iteration 0 and iteration 1: 193
Number of exact matches between iteration 0 and iteration 1: 142
Number of papers in iteration 0: 771
Number of papers in iteration 1: 771
----------------------------------------------


In [186]:
matches = 0
rough = 0
open_closed = [[0,0],[0,0]]
for paper in gemini_papers_iter0_comp.to_dict(orient="records"):
    #print(paper)
    for paper2 in gemini_papers_iter1.to_dict(orient="records"):
        if paper["Index"] == paper2["Index"] and paper["SourceID"] == paper2["SourceID"]:
            if paper["DOI"] == paper2["DOI"] and paper["DOI"] != "NO DOI":
                matches += 1
            open_closed[not paper["Open Access"]][not paper2["Open Access"]] += 1
        if paper["SourceID"] == paper2["SourceID"] and paper["DOI"] == paper2["DOI"] and paper["DOI"] != "NO DOI":
            rough += 1

print(matches)
print(rough)
print(open_closed)

fig = go.Figure(data=go.Heatmap(
                     z=[[open_closed[1][0], open_closed[1][1]], [open_closed[0][0], open_closed[0][1]]],
                     x=["Open Access", "Closed Access"],
                     y=["Open Access", "Closed Access"],
                     colorscale="Greens"),
                        layout=go.Layout(
                            title=f"Sample of Repeated Generations (Gemini 3.1 Flash Lite)",
                            xaxis_title="Iteration 1",
                            yaxis_title="Iteration 2"
                        ))
fig.update_layout(title=f"Sample of Repeated Generations (Gemini 3.1 Flash Lite)")

fig.update_traces(text=[[open_closed[1][0], open_closed[1][1]], [open_closed[0][0], open_closed[0][1]]], texttemplate="%{text}", textfont={"size": 20})
fig.update_layout(xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=["Open Access", "Closed Access"]),
                  yaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=["Open Access", "Closed Access"]))
fig.add_annotation(x=1.6, y=1, text=f"Total I2 OA: {open_closed[0][0] + open_closed[0][1]}", showarrow=False, font=dict(size=14), xanchor='left')
fig.add_annotation(x=1.6, y=0, text=f"Total I2 CA: {open_closed[1][0] + open_closed[1][1]}", showarrow=False, font=dict(size=14), xanchor='left')
fig.add_annotation(x=0, y=1.75, text=f"Total I1 OA: {open_closed[0][0] + open_closed[1][0]}", showarrow=False, font=dict(size=14), yanchor='bottom')
fig.add_annotation(x=1, y=1.75, text=f"Total I1 CA: {open_closed[0][1] + open_closed[1][1]}", showarrow=False, font=dict(size=14), yanchor='bottom')
fig.update_layout(plot_bgcolor = "white")
fig.show()
print(open_access_matrix)

142
193
[[208, 106], [120, 336]]


[[2248, 3325, 1106], [1582, 5651, 1196], [0, 0, 3]]


In [187]:
(open_closed[0][0] + open_closed[1][1]) / (open_closed[0][0] + open_closed[0][1] + open_closed[1][0] + open_closed[1][1])

0.7064935064935065

In [188]:
oa_gen_papers_iter1 = []
oa_gen_papers_iter1.append(chatgpt_papers_iter1[((chatgpt_papers_iter1["Open Access"] == True) | (chatgpt_papers_iter1["Open Access"] == False)) & (chatgpt_papers_iter1["Hallucination"] < hallucination_threshold)])
oa_gen_papers_iter1.append(claude_papers_iter1[((claude_papers_iter1["Open Access"] == True) | (claude_papers_iter1["Open Access"] == False)) & (claude_papers_iter1["Hallucination"] < hallucination_threshold)])
oa_gen_papers_iter1.append(gemini_papers_iter1[((gemini_papers_iter1["Open Access"] == True) | (gemini_papers_iter1["Open Access"] == False)) & (gemini_papers_iter1["Hallucination"] < hallucination_threshold)])

In [189]:
llms = ["ChatGPT 5.2", "Claude Opus 4.6", "Gemini 3.1 Flash Lite"]
combine_papers_llm_iter1 = [[],[],[]]
for i in range(3):
    base_llm = oa_gen_papers_iter0[i]
    comp_llm = oa_gen_papers_iter1[i]
    for base_paper in base_llm.to_dict("records"):
        comp_paper = comp_llm.loc[(comp_llm["Index"] == base_paper["Index"]) & (comp_llm["SourceID"] == base_paper["SourceID"])]
        if len(comp_paper) > 0:
            comp_paper = comp_paper.to_dict("records")[0]
            base_paper["Model"] = llms[i] + " Iter 0"
            comp_paper["Model"] = llms[i] + " Iter 1"
            base_paper["OA"] = base_paper["Open Access"]
            base_paper["OAS"] = base_paper["OA Standard"]
            comp_paper["OA"] = comp_paper["Open Access"]
            comp_paper["OAS"] = comp_paper["OA Standard"]
            combine_papers_llm_iter1[i].append(base_paper)
            combine_papers_llm_iter1[i].append(comp_paper)
    df = pd.DataFrame(combine_papers_llm_iter1[i])
    df_sub = df[["SourceID", "Model", "OA", "OAS", "Index"]]
    df_sub["SourceID"] = df_sub["SourceID"].astype(str)
    df_sub["Model"] = pd.Categorical(
        df_sub["Model"],
        categories=[llms[i] + " Iter 0", llms[i] + " Iter 1"]
    )
    df_sub["OA"] = df_sub["OA"].astype(int)
    df_sub.to_csv("../Sub_Sets/" + llms[i].split(" ")[0] + "_Iter0_Iter1.csv", index=False)

In [190]:
matches = 0
rough = 0
open_closed = [[0,0],[0,0]]
for paper in chatgpt_papers_iter0_comp.to_dict(orient="records"):
    #print(paper)
    for paper2 in chatgpt_papers_iter1.to_dict(orient="records"):
        if paper["Index"] == paper2["Index"] and paper["SourceID"] == paper2["SourceID"]:
            if paper["DOI"] == paper2["DOI"] and paper["DOI"] != "NO DOI":
                matches += 1
            open_closed[not paper["Open Access"]][not paper2["Open Access"]] += 1
        if paper["SourceID"] == paper2["SourceID"] and paper["DOI"] == paper2["DOI"] and paper["DOI"] != "NO DOI":
            rough += 1

print(matches)
print(rough)
print(open_closed)

fig = go.Figure(data=go.Heatmap(
                     z=[[open_closed[1][0], open_closed[1][1]], [open_closed[0][0], open_closed[0][1]]],
                     x=["Open Access", "Closed Access"],
                     y=["Open Access", "Closed Access"],
                     colorscale="Greens"),
                        layout=go.Layout(
                            title=f"Sample of Repeated Generations (ChatGPT 5.2)",
                            xaxis_title="Iteration 1",
                            yaxis_title="Iteration 2"
                        ))
fig.update_layout(title=f"Sample of Repeated Generations (ChatGPT 5.2)")

fig.update_traces(text=[[open_closed[1][0], open_closed[1][1]], [open_closed[0][0], open_closed[0][1]]], texttemplate="%{text}", textfont={"size": 20})
fig.update_layout(xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=["Open Access", "Closed Access"]),
                  yaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=["Open Access", "Closed Access"]))
fig.add_annotation(x=1.6, y=1, text=f"Total I2 OA: {open_closed[0][0] + open_closed[0][1]}", showarrow=False, font=dict(size=14), xanchor='left')
fig.add_annotation(x=1.6, y=0, text=f"Total I2 CA: {open_closed[1][0] + open_closed[1][1]}", showarrow=False, font=dict(size=14), xanchor='left')
fig.add_annotation(x=0, y=1.75, text=f"Total I1 OA: {open_closed[0][0] + open_closed[1][0]}", showarrow=False, font=dict(size=14), yanchor='bottom')
fig.add_annotation(x=1, y=1.75, text=f"Total I1 CA: {open_closed[0][1] + open_closed[1][1]}", showarrow=False, font=dict(size=14), yanchor='bottom')
fig.update_layout(plot_bgcolor = "white")
fig.show()
print(open_access_matrix)

242
324
[[178, 105], [87, 431]]


[[2248, 3325, 1106], [1582, 5651, 1196], [0, 0, 3]]


In [191]:
(open_closed[0][0] + open_closed[1][1]) / (open_closed[0][0] + open_closed[0][1] + open_closed[1][0] + open_closed[1][1])

0.7602996254681648

In [192]:
matches = 0
rough = 0
open_closed = [[0,0],[0,0]]
for paper in claude_papers_iter0_comp.to_dict(orient="records"):
    #print(paper)
    for paper2 in claude_papers_iter1.to_dict(orient="records"):
        if paper["Index"] == paper2["Index"] and paper["SourceID"] == paper2["SourceID"]:
            if paper["DOI"] == paper2["DOI"] and paper["DOI"] != "NO DOI":
                matches += 1
            open_closed[not paper["Open Access"]][not paper2["Open Access"]] += 1
        if paper["SourceID"] == paper2["SourceID"] and paper["DOI"] == paper2["DOI"] and paper["DOI"] != "NO DOI":
            rough += 1

print(matches)
print(rough)
print(open_closed)

fig = go.Figure(data=go.Heatmap(
                     z=[[open_closed[1][0], open_closed[1][1]], [open_closed[0][0], open_closed[0][1]]],
                     x=["Open Access", "Closed Access"],
                     y=["Open Access", "Closed Access"],
                     colorscale="Greens"),
                        layout=go.Layout(
                            title=f"Sample of Repeated Generations (Claude Opus 4.6)",
                            xaxis_title="Iteration 1",
                            yaxis_title="Iteration 2"
                        ))
fig.update_layout(title=f"Sample of Repeated Generations (Claude Opus 4.6)")

fig.update_traces(text=[[open_closed[1][0], open_closed[1][1]], [open_closed[0][0], open_closed[0][1]]], texttemplate="%{text}", textfont={"size": 20})
fig.update_layout(xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=["Open Access", "Closed Access"]),
                  yaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=["Open Access", "Closed Access"]))
fig.add_annotation(x=1.6, y=1, text=f"Total I2 OA: {open_closed[0][0] + open_closed[0][1]}", showarrow=False, font=dict(size=14), xanchor='left')
fig.add_annotation(x=1.6, y=0, text=f"Total I2 CA: {open_closed[1][0] + open_closed[1][1]}", showarrow=False, font=dict(size=14), xanchor='left')
fig.add_annotation(x=0, y=1.75, text=f"Total I1 OA: {open_closed[0][0] + open_closed[1][0]}", showarrow=False, font=dict(size=14), yanchor='bottom')
fig.add_annotation(x=1, y=1.75, text=f"Total I1 CA: {open_closed[0][1] + open_closed[1][1]}", showarrow=False, font=dict(size=14), yanchor='bottom')
fig.update_layout(plot_bgcolor = "white")
fig.show()
print(open_access_matrix)

314
359
[[226, 100], [85, 390]]


[[2248, 3325, 1106], [1582, 5651, 1196], [0, 0, 3]]


In [193]:
(open_closed[0][0] + open_closed[1][1]) / (open_closed[0][0] + open_closed[0][1] + open_closed[1][0] + open_closed[1][1])

0.7690387016229713

In [194]:
chatgpt_papers_iter2 = []
claude_papers_iter2 = []
gemini_papers_iter2 = []

for (source_id, llm), gen in generated_papers_iter2.items():
    gen_paper_iter2 = generated_papers_iter2.get((source_id, llm), [])
    gen_paper_iter2 = pd.DataFrame(gen_paper_iter2)
    gen_paper_iter2["SourceID"] = source_id
    gen_paper_iter2 = gen_paper_iter2.to_dict(orient="records")
    if llm == "ChatGPT":
        chatgpt_papers_iter2.extend(gen_paper_iter2)
    elif llm == "Claude":
        claude_papers_iter2.extend(gen_paper_iter2)
    elif llm == "Gemini":
        gemini_papers_iter2.extend(gen_paper_iter2)

chatgpt_papers_iter2 = pd.DataFrame(chatgpt_papers_iter2)
claude_papers_iter2 = pd.DataFrame(claude_papers_iter2)
gemini_papers_iter2 = pd.DataFrame(gemini_papers_iter2)

In [195]:
stacked_data = []
for llm, df in [("ChatGPT 5.2", chatgpt_papers_iter2), ("Claude Opus 4.6", claude_papers_iter2), ("Gemini 3.1 Flash Lite", gemini_papers_iter2)]:
    oa_counts = df[(df["Hallucination"] <= hallucination_threshold) & (df["Hallucination"] != -1)]["Open Access"].value_counts()
    hallucination_count = df[df["Hallucination"] > hallucination_threshold].shape[0]
    no_doi = df[df["Hallucination"] == -1].shape[0]
    stacked_data.append({
        "LLM": llm,
        "Open Access": oa_counts.get(True, 0),
        "Closed Access": oa_counts.get(False, 0),
        "Hallucination": hallucination_count,
        "No DOI": no_doi
    })

ground_oa_counts = papers_second_iter["Open Access"].value_counts()

stacked_data.insert(0, {
    "LLM": "Ground Truth",
    "Open Access": ground_oa_counts.get(True, 0),
    "Closed Access": ground_oa_counts.get(False, 0),
    "No DOI": papers_second_iter.shape[0] - ground_oa_counts.sum(),
    "Hallucination": 0
})

stacked_df = pd.DataFrame(stacked_data)

fig = px.bar(stacked_df, x="LLM", y=["Open Access", "Closed Access", "Hallucination", "No DOI"], title="Open Access Status by LLM", subtitle=f"Hallucination Threshold = {hallucination_threshold}, iteration with biased query", labels={"value": "Count", "LLM": "Source"}, text_auto=True, color_discrete_map={"Open Access": "lightgreen", "Closed Access": "salmon", "Hallucination": "lightblue", "No DOI": "lightgray"}, height=500)
fig.show()

print(stacked_df)

                     LLM  Open Access  Closed Access  No DOI  Hallucination
0           Ground Truth          334            379      88              0
1            ChatGPT 5.2          215            311     101            174
2        Claude Opus 4.6          225            339      93            144
3  Gemini 3.1 Flash Lite          111            203     126            361


In [196]:
stacked_df_oa_ca = stacked_df[["LLM", "Open Access", "Closed Access"]]
stacked_df_oa_ca = stacked_df_oa_ca.melt(id_vars=["LLM"], value_vars=["Open Access", "Closed Access"], var_name="Status", value_name="Count")

stacked_df_oa_ca["Count"] = stacked_df_oa_ca.groupby("LLM")["Count"].transform(lambda x: x / x.sum() * 100).round(2)
fig = px.bar(stacked_df_oa_ca, x="LLM", y="Count", color="Status", title="Open Access vs Closed Access by LLM", subtitle=f"Hallucination Threshold = {hallucination_threshold}, iteration with biased query", labels={"Count": "Percentage", "LLM": "Source"}, text_auto=True, color_discrete_map={"Open Access": "lightgreen", "Closed Access": "salmon"})
fig.show()



In [197]:
stacked_data_oa_standard = []
for llm, df in [("ChatGPT 5.2", chatgpt_papers_iter2), ("Claude Opus 4.6", claude_papers_iter2), ("Gemini 3.1 Flash Lite", gemini_papers_iter2)]:
    oa_df = df[0 <= (df["Hallucination"] > hallucination_threshold) & (df["Open Access"] == True)]
    oa_standard_counts = oa_df["OA Standard"].value_counts()
    hallucination_count = df[df["Hallucination"] > hallucination_threshold].shape[0]
    stacked_data_oa_standard.append({
        "LLM": llm,
        "Gold": oa_standard_counts.get("gold", 0),
        "Green": oa_standard_counts.get("green", 0),
        "Bronze": oa_standard_counts.get("bronze", 0),
        "Hybrid": oa_standard_counts.get("hybrid", 0)
    })

ground_oa_standard_counts = papers_second_iter[(papers_second_iter["Open Access"] == True)]["OA Standard"].value_counts()
stacked_data_oa_standard.insert(0, {
    "LLM": "Ground Truth",
    "Gold": ground_oa_standard_counts.get("gold", 0),
    "Green": ground_oa_standard_counts.get("green", 0),
    "Bronze": ground_oa_standard_counts.get("bronze", 0),
    "Hybrid": ground_oa_standard_counts.get("hybrid", 0)
})
stacked_df_oa_standard = pd.DataFrame(stacked_data_oa_standard)

fig = px.bar(stacked_df_oa_standard, x="LLM", y=["Gold", "Green", "Bronze", "Hybrid"], title="OA Standard by LLM (Only for OA Papers)", subtitle=f"Hallucination Threshold = {hallucination_threshold}", labels={"value": "Count", "LLM": "Source", "variable": "Standard"}, text_auto=True, color_discrete_map={"Gold": "gold", "Green": "green", "Bronze": "peru", "Hybrid": "lightblue"})
fig.update_layout(
    legend=dict(
        title=dict(
            text="Standard"
        )
    )
)
fig.show()
print(stacked_df_oa_standard)

oa_cols = ["Gold", "Green", "Bronze", "Hybrid"]

stacked_df_oa_standard_pct = stacked_df_oa_standard.copy()
stacked_df_oa_standard_pct[oa_cols] = (
    stacked_df_oa_standard_pct[oa_cols]
    .div(stacked_df_oa_standard_pct[oa_cols].sum(axis=1), axis=0)
    * 100
)

fig = px.bar(
    stacked_df_oa_standard_pct,
    x="LLM",
    y=oa_cols,
    title="OA Standard Distribution by LLM (Only OA Papers)",
    subtitle=f"Hallucination Threshold = {hallucination_threshold}",
    labels={"value": "Percentage (%)", "LLM": "Source", "variable": "Standard"},
    text_auto=".2f",
    color_discrete_map={
        "Gold": "gold",
        "Green": "green",
        "Bronze": "peru",
        "Hybrid": "lightblue"
    }
)

fig.update_layout(
    yaxis_title="Percentage (%)",
    yaxis_range=[0, 110],
    legend=dict(
        title=dict(
            text="Standard"
        )
    )
)

fig.show()

                     LLM  Gold  Green  Bronze  Hybrid
0           Ground Truth   104    106      63      61
1            ChatGPT 5.2    68    105      78      13
2        Claude Opus 4.6    61    105      64      32
3  Gemini 3.1 Flash Lite    54     74      44      14


In [198]:
combine_papers_iter2 = [[], [], []]
for llm, df, x in [("ChatGPT 5.2", chatgpt_papers_iter2, 0), ("Claude Opus 4.6", claude_papers_iter2, 1),
                   ("Gemini 3.1 Flash Lite", gemini_papers_iter2, 2)]:
    for oa_paper in oa_ref_papers.to_dict("records"):
        gen_paper = df.loc[(df["Index"] == oa_paper["Index"]) & (df["SourceID"] == oa_paper["SourceID"]) & (
                    df["Hallucination"] < hallucination_threshold)]
        if len(gen_paper) > 0:
            gen_paper = gen_paper.to_dict("records")[0]
            if isinstance(gen_paper["Open Access"], bool):
                gen_paper["Model"] = llm
                gen_paper["OA"] = gen_paper["Open Access"]
                gen_paper["OAS"] = gen_paper["OA Standard"]
                oa_paper["OA"] = oa_paper["Open Access"]
                oa_paper["OAS"] = oa_paper["OA Standard"]
                combine_papers_iter2[x].append(gen_paper)
                combine_papers_iter2[x].append(oa_paper)

for llm, x in [("ChatGPT 5.2", 0), ("Claude Opus 4.6", 1), ("Gemini 3.1 Flash Lite", 2)]:
    df = pd.DataFrame(combine_papers_iter2[x])
    df_sub = df[["SourceID", "Model", "OA", "OAS", "Index"]]
    df_sub["SourceID"] = df_sub["SourceID"].astype(str)
    df_sub["Model"] = pd.Categorical(
        df_sub["Model"],
        categories=["Ground Truth", llm]
    )
    df_sub["OA"] = df_sub["OA"].astype(int)
    df_sub.to_csv("../Sub_Sets/Ground-" + llm.split(" ")[0] + "_Iter2.csv", index=False)

In [199]:
oa_gen_papers_iter2 = []
oa_gen_papers_iter2.append(chatgpt_papers_iter2[((chatgpt_papers_iter2["Open Access"] == True) | (chatgpt_papers_iter2["Open Access"] == False)) & (chatgpt_papers_iter2["Hallucination"] < hallucination_threshold)])
oa_gen_papers_iter2.append(claude_papers_iter2[((claude_papers_iter2["Open Access"] == True) | (claude_papers_iter2["Open Access"] == False)) & (claude_papers_iter2["Hallucination"] < hallucination_threshold)])
oa_gen_papers_iter2.append(gemini_papers_iter2[((gemini_papers_iter2["Open Access"] == True) | (gemini_papers_iter2["Open Access"] == False)) & (gemini_papers_iter2["Hallucination"] < hallucination_threshold)])

In [200]:
llms = ["ChatGPT 5.2", "Claude Opus 4.6", "Gemini 3.1 Flash Lite"]
combine_papers_llm_iter2 = [[],[],[]]
for i in range(3):
    base_llm = oa_gen_papers_iter0[i]
    comp_llm = oa_gen_papers_iter2[i]
    for base_paper in base_llm.to_dict("records"):
        comp_paper = comp_llm.loc[(comp_llm["Index"] == base_paper["Index"]) & (comp_llm["SourceID"] == base_paper["SourceID"])]
        if len(comp_paper) > 0:
            comp_paper = comp_paper.to_dict("records")[0]
            base_paper["Model"] = llms[i] + " Iter 0"
            comp_paper["Model"] = llms[i] + " Iter 1"
            base_paper["OA"] = base_paper["Open Access"]
            base_paper["OAS"] = base_paper["OA Standard"]
            comp_paper["OA"] = comp_paper["Open Access"]
            comp_paper["OAS"] = comp_paper["OA Standard"]
            combine_papers_llm_iter2[i].append(base_paper)
            combine_papers_llm_iter2[i].append(comp_paper)
    df = pd.DataFrame(combine_papers_llm_iter2[i])
    df_sub = df[["SourceID", "Model", "OA", "OAS", "Index"]]
    df_sub["SourceID"] = df_sub["SourceID"].astype(str)
    df_sub["Model"] = pd.Categorical(
        df_sub["Model"],
        categories=[llms[i] + " Iter 0", llms[i] + " Iter 1"]
    )
    df_sub["OA"] = df_sub["OA"].astype(int)
    df_sub.to_csv("../Sub_Sets/" + llms[i].split(" ")[0] + "_Iter0_Iter2.csv", index=False)